# Width sweeps on Kaggle

Runs sweeps of `width_sweep.py` (see `docs/WIDTH-QUESTIONS.md`) or `sanford_repro.py`
(the Sanford et al. 2024a reproduction) as parallel jobs on the notebook's GPU(s).

**Settings (right panel):** Accelerator = GPU T4 x2 (recent PyTorch builds dropped P100 support), Internet = On.
Run it unattended with **Save Version → Save & Run All (Commit)**, then download
`<results>_results.zip` (`width_results.zip` or `sanford_results.zip`) from the version's Output tab.

In [ ]:
SCRIPT = "width_sweep.py"     # or "sanford_repro.py"
SWEEPS = ["ebd12hd2_128"]  # longest jobs first
SHARDS = 4                    # workers per sweep; width_sweep's tiny models share a GPU 8 ways
MAX_PAR = 4                   # workers alive at once; n = 128 holds ~4 GB per worker, 2 fit a T4
# Keep len(SWEEPS) * SHARDS <= ~8: every worker holds its training set in RAM (128k graphs:
# 0.26 GB at n = 32, 1.3 GB at n = 72) and Kaggle has 30 GB. 64 workers on 2026-10-19 ran
# out of memory (38 killed with -9).
BRANCH = "master"

import os
os.environ["WIDTH_CACHE"] = "/tmp/width_cache"   # datasets cache, kept out of the output zip
RESULTS = "width" if SCRIPT == "width_sweep.py" else "sanford"

import time
SESSION_START = time.time()   # Kaggle stops a session at 12 h, cutting off later cells
DEADLINE = SESSION_START + 10 * 3600   # all training stops here; leaves time to analyze and zip
STOPPED = False


In [ ]:
!git clone -q --depth 1 -b $BRANCH https://github.com/barsboldb/mpgnn.git
%cd mpgnn
!pip install -q torch-geometric
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git log -1 --format='%h %s'


In [ ]:
# Build every dataset the sweeps need (no-op for sweeps without cached data).
for sw in SWEEPS:
    !python -u $SCRIPT {sw} --prepare


In [ ]:
import os, subprocess, time, torch

gpus = max(1, torch.cuda.device_count())
out = f"results/{RESULTS}"
os.makedirs(out, exist_ok=True)
jobs = [(sw, i) for sw in SWEEPS for i in range(SHARDS)]
cap = globals().get("MAX_PAR", len(jobs))   # queue: at most cap workers alive at once
procs, slots = [], {}                       # slots: GPU of each live worker
def launch():
    while jobs and sum(p.poll() is None for p in procs) < cap:
        busy = [slots[p] for p in procs if p.poll() is None]
        gpu = min(range(gpus), key=busy.count)
        sw, i = jobs.pop(0)
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        log = open(f"{out}/{sw}.shard{i}.log", "w")
        p = subprocess.Popen(["python", "-u", SCRIPT, sw, "--shard", f"{i}/{SHARDS}"],
                             stdout=log, stderr=subprocess.STDOUT, env=env)
        procs.append(p); slots[p] = gpu
launch()
print(f"{len(procs)} jobs started on {gpus} GPU(s), {len(jobs)} queued")

t0 = time.time()
def snapshot():   # zip what is on disk, so a killed session still leaves results behind
    subprocess.run(f"cd results && zip -qr /kaggle/working/{RESULTS}_results.zip {RESULTS}",
                   shell=True)
last_zip = time.time()
while jobs or any(p.poll() is None for p in procs):
    time.sleep(120)
    if not STOPPED:
        launch()
    if time.time() - last_zip > 1800:
        snapshot(); last_zip = time.time()
    if time.time() > DEADLINE and not STOPPED:
        for p in procs:
            p.terminate()
        jobs.clear()
        STOPPED = True
        print("deadline reached: workers stopped", flush=True)
    done = sum(1 for f in os.listdir(out) if f.endswith(".jsonl") and f.split(".")[0] in SWEEPS
               for _ in open(f"{out}/{f}"))
    print(f"{(time.time() - t0) / 60:5.1f} min  runs saved: {done}", flush=True)
print("exit codes:", [p.returncode for p in procs])


In [ ]:
# Catch anything a job missed or crashed on (no-op when complete), then analyze.
# Bounded by DEADLINE: on 2026-10-19 this cell ran unbounded after the workers died and
# Kaggle killed the session at 12 h, before the zip.
for sw in SWEEPS:
    left = DEADLINE - time.time()
    if not STOPPED and left > 600:
        try:
            subprocess.run(["python", "-u", SCRIPT, sw], timeout=left)
        except subprocess.TimeoutExpired:
            STOPPED = True
            print(f"deadline reached during the catch-up of {sw}", flush=True)
    !python $SCRIPT {sw} --analyze
    if SCRIPT == "width_sweep.py":
        !python $SCRIPT {sw} --curves


In [ ]:
# Layer probes on the weights a sweep saved (sweeps with save_models, e.g. "probe");
# results/width/probe/*.json go into the zip. Minutes on the GPU.
if SCRIPT == "width_sweep.py" and "probe" in SWEEPS and time.time() < DEADLINE:
    left = DEADLINE + 1800 - time.time()   # may use half of the analysis hour
    try:
        subprocess.run(["python", "-u", "probe_layers.py"], timeout=left)
    except subprocess.TimeoutExpired:
        print("probing stopped at the deadline; finished models are saved", flush=True)


In [ ]:
!cd results && zip -qr /kaggle/working/{RESULTS}_results.zip {RESULTS} && ls -lh /kaggle/working/{RESULTS}_results.zip
